In [ ]:
import re
from pathlib import Path
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text_parts = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text()
            if page_text:
                text_parts.append(f"[[SEITE {page_number}]]\n{page_text}")

    return "\n\n".join(text_parts)

def clean_pdf_text(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            cleaned_lines.append("")
            continue

        if re.fullmatch(r"\d+", line):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

def pdf_to_txt_path(pdf_path, pdf_root, txt_root):
    relative_path = pdf_path.relative_to(pdf_root)
    txt_relative = relative_path.with_suffix(".txt")
    return txt_root / txt_relative

def export_pdfs_to_txt(pdf_root, txt_root, skip_existing: bool = True):
    pdf_root = Path(pdf_root)
    txt_root = Path(txt_root)

    results = []

    for pdf_path in pdf_root.rglob("*.pdf"):
        txt_path = pdf_to_txt_path(pdf_path, pdf_root, txt_root)
        txt_path.parent.mkdir(parents=True, exist_ok=True)

        if skip_existing and txt_path.exists():
            results.append({
                "status": "skipped",
                "pdf_datei": str(pdf_path),
                "txt_datei": str(txt_path),
                "fehler": None
            })
            continue

        try:
            text = extract_text_from_pdf(pdf_path)
            cleaned_text = clean_pdf_text(text)

            txt_path.write_text(cleaned_text, encoding="utf-8")

            results.append({
                "status": "ok",
                "pdf_datei": str(pdf_path),
                "txt_datei": str(txt_path),
                "fehler": None
            })

        except Exception as e:
            results.append({
                "status": "fehler",
                "pdf_datei": str(pdf_path),
                "txt_datei": str(txt_path),
                "fehler": str(e)
            })

    return results

export_pdfs_to_txt(pdf_root="../00_sources/pdfs", txt_root="txt1")

In [ ]:
import re
from pathlib import Path
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
german_stopwords = set(stopwords.words("german"))

PAGE_MARKER_PATTERN = re.compile(r"\[\[SEITE\s+(\d+)\]\]")

def split_text_into_pages(text):
    parts = PAGE_MARKER_PATTERN.split(text)

    pages = []
    if not parts:
        return pages

    if parts[0].strip():
        pages.append({
            "page_number": 1,
            "text": parts[0]
        })

    i = 1
    while i < len(parts) - 1:
        page_number = int(parts[i])
        page_text = parts[i + 1]
        pages.append({
            "page_number": page_number,
            "text": page_text
        })
        i += 2

    return pages

def preprocess_with_positions(text):
    lowered = text.lower()
    cleaned = re.sub(r"[^a-zäöüß\s]", " ", lowered)
    raw_tokens = cleaned.split()

    tokens = []
    for idx, token in enumerate(raw_tokens):
        if token in german_stopwords:
            continue
        if len(token) <= 2:
            continue

        tokens.append({
            "token": token,
            "position": idx
        })

    return tokens

def path_metadata_fixed_depth(file_path, root_folder, max_depth=3):
    rel_path = file_path.relative_to(root_folder)
    parts = list(rel_path.parts[:-1])

    data = {
        "datei": rel_path.name,
        "relativer_pfad": str(rel_path),
        "voller_pfad": str(file_path)
    }

    for i in range(max_depth):
        data[f"ordner_{i+1}"] = parts[i] if i < len(parts) else None

    return data

def find_keyword_occurrences(tokens_with_pos, keywords, window_size=5):
    keyword_set = set(k.lower() for k in keywords)
    rows = []
    plain_tokens = [item["token"] for item in tokens_with_pos]

    for i, item in enumerate(tokens_with_pos):
        token = item["token"]

        if token in keyword_set:
            snippet_start = max(0, i - 10)
            snippet_end = min(len(tokens_with_pos), i + 11)
            snippet = " ".join(plain_tokens[snippet_start:snippet_end])

            start = max(0, i - window_size)
            end = min(len(tokens_with_pos), i + window_size + 1)

            for j in range(start, end):
                if j == i:
                    continue

                context_item = tokens_with_pos[j]

                rows.append({
                    "schlagwort": token,
                    "schlagwort_position": item["position"],
                    "kontextwort": context_item["token"],
                    "kontext_position": context_item["position"],
                    "offset": j - i,
                    "ausschnitt": snippet
                })

    return rows

def analyze_txt_files(txt_root, keywords, window_size=5, max_depth=3):
    txt_root = Path(txt_root)

    fundstellen_rows = []
    dokumente_rows = []

    for txt_path in txt_root.rglob("*.txt"):
        meta = path_metadata_fixed_depth(txt_path, txt_root, max_depth=max_depth)

        try:
            text = txt_path.read_text(encoding="utf-8")
            pages = split_text_into_pages(text)

            total_hits = 0
            total_context_rows = 0

            for page in pages:
                tokens_with_pos = preprocess_with_positions(page["text"])
                occurrence_rows = find_keyword_occurrences(tokens_with_pos, keywords, window_size)

                keyword_set = set(k.lower() for k in keywords)
                page_hits = sum(1 for t in tokens_with_pos if t["token"] in keyword_set)

                total_hits += page_hits
                total_context_rows += len(occurrence_rows)

                for row in occurrence_rows:
                    fundstellen_rows.append({
                        **meta,
                        "seite": page["page_number"],
                        **row
                    })

            dokumente_rows.append({
                **meta,
                "status": "ok",
                "seiten_ausgelesen": len(pages),
                "schlagwort_treffer": total_hits,
                "kontext_fundstellen": total_context_rows,
                "fehler": None
            })

        except Exception as e:
            dokumente_rows.append({
                **meta,
                "status": "fehler",
                "seiten_ausgelesen": None,
                "schlagwort_treffer": None,
                "kontext_fundstellen": None,
                "fehler": str(e)
            })

    fundstellen_df = pd.DataFrame(fundstellen_rows)
    dokumente_df = pd.DataFrame(dokumente_rows)

    return fundstellen_df, dokumente_df


keywords = ["programmierwerkzeug", "programmiersprache", "programmierumgebung", "ide"]
pt = ["python", "java", "pascal", "javascript", "scratch", "scheme", "lisp", "blockly", "c", "c#", "c++" "calliope", "delphi", "php", "visualbasic", "prolog", "haskell"]

fundstellen_df, dokumente_df = analyze_txt_files(
    txt_root="txt1",  # early keyword pass; its old output is in ../legacy/early_keyword_pass/csv
    keywords=keywords,
    window_size=5,
    max_depth=6
)

fundstellen_df.to_csv("csv/fundstellen.csv", index=False)
dokumente_df.to_csv("csv/dokumente.csv", index=False)


In [ ]:
fundstellen_df

In [ ]:
dokumente_df